In [3]:
"""
Data Mart CEMEX - FACT_DESPACHOS (modelo estrella)

Lee Dataset_CEMEX.xlsx (hoja "Data", ~578 mil filas x 121 columnas), crea las tablas
del datamart en SQL Server y las puebla.

 - Origen : Dataset_CEMEX.xlsx
 - Destino: base de datos CemexDW en SQL Server (se crea si no existe)

Regla clave: FACT_DESPACHOS tiene EXACTAMENTE una fila por cada fila del Excel
(id_despacho = n° de fila del dataset, empezando en 1). El script lo valida y
se detiene con error si no coincide.

"""

import os
import sys
import time
import zipfile
import datetime as dt

import numpy as np
import pandas as pd
import pyodbc

try:                                    # lxml es opcional (más rápido)
    from lxml import etree as ET
except ImportError:                     # si no está, se usa la librería estándar
    import xml.etree.ElementTree as ET

# ============================== CONFIGURACIÓN ==============================
try:
    _CARPETA = os.path.dirname(os.path.abspath(__file__))   # ejecutado como .py
except NameError:
    _CARPETA = os.getcwd()                                  # notebook (Jupyter/Colab)
# Si el Excel está en otra carpeta, pon aquí la ruta completa, ej: r"C:\Users\tu_usuario\Desktop\Dataset_CEMEX.xlsx"
EXCEL_PATH = os.path.join(_CARPETA, "Dataset_CEMEX.xlsx")
SERVER = "localhost"
UID = "sa"
PWD = "1234567890"
DB_DESTINO = "CemexDW"

# Leer el Excel de 370 MB tarda varios minutos. Con USAR_CACHE=True se guarda una copia
# rápida (.pkl) junto al Excel y las siguientes ejecuciones tardan segundos.
# Se vuelve a leer el Excel automáticamente si el Excel es más nuevo que la copia.
USAR_CACHE = True
CACHE_PATH = EXCEL_PATH + ".cache.pkl"

BATCH = 20000                           # filas por lote de INSERT
# ===========================================================================

# Columnas del Excel que usa el datamart (el resto de las 121 se ignora)
COLUMNAS = [
    "iddia", "idoficina", "idmaterialcom", "idtipodespacho", "idcliente", "idestadoentrega",
    "idruta", "idunidadmedida", "idclasedespacho", "idsubcanal", "idempresatransporte",
    "nroentrega", "nropedido", "nrofactura", "nroguiaremision", "placa",
    "ctdpedida", "ctddespachada", "estadoalmacen", "idtipovehiculo",
    "vtadespachada", "vtadespachadabruto", "descuento", "importefletefactura",
    "codvendedorinterno", "vendedorinterno", "idgrupovendedor", "iddestinatario", "destinofinal",
]
COLS_ENTERAS = [
    "iddia", "idoficina", "idmaterialcom", "idtipodespacho", "idcliente", "idestadoentrega",
    "idruta", "idunidadmedida", "idclasedespacho", "idsubcanal", "idempresatransporte",
    "estadoalmacen", "idtipovehiculo", "codvendedorinterno", "idgrupovendedor", "iddestinatario",
]
MEDIDAS = ["ctdpedida", "ctddespachada", "vtadespachada", "vtadespachadabruto",
           "descuento", "importefletefactura"]
DEGENERADAS = ["nroentrega", "nropedido", "nrofactura", "nroguiaremision"]


# ------------------------------------------------------------------ conexión
def _driver():
    """Elige el ODBC Driver más nuevo de SQL Server instalado (17 en el curso)."""
    drivers = [d for d in pyodbc.drivers() if "ODBC Driver" in d and "SQL Server" in d]
    if not drivers:
        raise RuntimeError("No hay 'ODBC Driver xx for SQL Server' instalado.")
    return sorted(drivers, key=lambda d: int(d.split()[2]))[-1]


def get_connection(database, autocommit=False):
    try:
        conn = pyodbc.connect(
            f"DRIVER={{{_driver()}}};SERVER={SERVER};DATABASE={database};"
            f"UID={UID};PWD={PWD};TrustServerCertificate=yes",
            autocommit=autocommit,
        )
        print(f"✓ Conexión a {database} exitosa")
        return conn
    except pyodbc.Error as e:
        print(f"Error de conexión a {database}: {e}")
        return None


def crear_base_si_no_existe():
    conn = get_connection("master", autocommit=True)
    if conn is None:
        sys.exit(1)
    conn.cursor().execute(f"IF DB_ID('{DB_DESTINO}') IS NULL CREATE DATABASE [{DB_DESTINO}]")
    conn.close()


# ------------------------------------------------------------- esquema (DDL)
# Orden de borrado: primero la fact (tiene las FK), luego las dimensiones.
TABLAS_DROP = ["FACT_DESPACHOS", "DIM_TIEMPO", "DIM_CLIENTE", "DIM_DESTINATARIO", "DIM_PRODUCTO",
               "DIM_OFICINA", "DIM_RUTA", "DIM_TRANSPORTE", "DIM_VENDEDOR", "DIM_CANAL",
               "DIM_TIPO_DESPACHO", "DIM_UNIDAD_MEDIDA"]

DDL = [
    """CREATE TABLE DIM_TIEMPO (
        id_tiempo   INT PRIMARY KEY,
        iddia       DATE NOT NULL UNIQUE,
        anio        INT NOT NULL,
        trimestre   INT NOT NULL,
        mes         INT NOT NULL,
        semana      INT NOT NULL,
        dia_semana  NVARCHAR(15) NOT NULL
    )""",
    """CREATE TABLE DIM_CLIENTE (
        id_cliente    INT PRIMARY KEY,
        idcliente     INT NOT NULL,
        tipo_cliente  NVARCHAR(50) NULL,
        segmento      NVARCHAR(50) NULL
    )""",
    """CREATE TABLE DIM_DESTINATARIO (
        id_destinatario     INT PRIMARY KEY,
        iddestinatario      INT NOT NULL,
        cod_destinatario    NVARCHAR(50) NULL,
        nombre_destinatario NVARCHAR(200) NULL
    )""",
    """CREATE TABLE DIM_PRODUCTO (
        id_producto    INT PRIMARY KEY,
        idmaterialcom  INT NOT NULL,
        desmaterialcom NVARCHAR(200) NULL,
        idunidadmedida INT NOT NULL,
        familia        NVARCHAR(100) NULL,
        linea          NVARCHAR(100) NULL
    )""",
    """CREATE TABLE DIM_OFICINA (
        id_oficina     INT PRIMARY KEY,
        idoficina      INT NOT NULL,
        nombre_oficina NVARCHAR(100) NULL,
        zona           NVARCHAR(100) NULL
    )""",
    """CREATE TABLE DIM_RUTA (
        id_ruta          INT PRIMARY KEY,
        idruta           INT NOT NULL,
        descripcion_ruta NVARCHAR(200) NULL,
        zona_destino     NVARCHAR(100) NULL
    )""",
    """CREATE TABLE DIM_TRANSPORTE (
        id_transporte       INT PRIMARY KEY,
        idempresatransporte INT NOT NULL,
        nombre_empresa      NVARCHAR(200) NULL,
        idtipovehiculo      INT NOT NULL,
        placa               NVARCHAR(20) NOT NULL
    )""",
    """CREATE TABLE DIM_VENDEDOR (
        id_vendedor        INT PRIMARY KEY,
        codvendedorinterno INT NOT NULL,
        vendedorinterno    NVARCHAR(150) NOT NULL,
        idgrupovendedor    INT NOT NULL
    )""",
    """CREATE TABLE DIM_CANAL (
        id_canal   INT PRIMARY KEY,
        idsubcanal INT NOT NULL,
        desc_canal NVARCHAR(100) NULL
    )""",
    """CREATE TABLE DIM_TIPO_DESPACHO (
        id_tipo_despacho INT PRIMARY KEY,
        idtipodespacho   INT NOT NULL,
        idclasedespacho  INT NOT NULL,
        idestadoentrega  INT NOT NULL,
        estadoalmacen    NVARCHAR(20) NOT NULL
    )""",
    """CREATE TABLE DIM_UNIDAD_MEDIDA (
        id_unidad_medida INT PRIMARY KEY,
        idunidadmedida   INT NOT NULL,
        desc_unidad      NVARCHAR(50) NULL
    )""",
    """CREATE TABLE FACT_DESPACHOS (
        id_despacho      INT PRIMARY KEY,
        id_tiempo        INT NOT NULL REFERENCES DIM_TIEMPO(id_tiempo),
        id_cliente       INT NOT NULL REFERENCES DIM_CLIENTE(id_cliente),
        id_destinatario  INT NOT NULL REFERENCES DIM_DESTINATARIO(id_destinatario),
        id_producto      INT NOT NULL REFERENCES DIM_PRODUCTO(id_producto),
        id_oficina       INT NOT NULL REFERENCES DIM_OFICINA(id_oficina),
        id_ruta          INT NOT NULL REFERENCES DIM_RUTA(id_ruta),
        id_transporte    INT NOT NULL REFERENCES DIM_TRANSPORTE(id_transporte),
        id_vendedor      INT NOT NULL REFERENCES DIM_VENDEDOR(id_vendedor),
        id_canal         INT NOT NULL REFERENCES DIM_CANAL(id_canal),
        id_tipo_despacho INT NOT NULL REFERENCES DIM_TIPO_DESPACHO(id_tipo_despacho),
        id_unidad_medida INT NOT NULL REFERENCES DIM_UNIDAD_MEDIDA(id_unidad_medida),
        nroentrega       NVARCHAR(30) NULL,
        nropedido        NVARCHAR(30) NULL,
        nrofactura       NVARCHAR(30) NULL,
        nroguiaremision  NVARCHAR(40) NULL,
        ctdpedida           DECIMAL(18,4) NULL,
        ctddespachada       DECIMAL(18,4) NULL,
        vtadespachada       DECIMAL(18,4) NULL,
        vtadespachadabruto  DECIMAL(18,4) NULL,
        descuento           DECIMAL(18,4) NULL,
        importefletefactura DECIMAL(18,4) NULL
    )""",
]


def crear_esquema(conn):
    cur = conn.cursor()
    for t in TABLAS_DROP:               # re-ejecutable: se recrea todo desde cero
        cur.execute(f"DROP TABLE IF EXISTS {t}")
    for ddl in DDL:
        cur.execute(ddl)
    conn.commit()
    print("✓ Esquema del Data Mart creado (11 dimensiones + FACT_DESPACHOS).")


# ------------------------------------------------------- lectura del Excel
_NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"


def _col_idx(ref):
    n = 0
    for ch in ref:
        if not ch.isalpha():
            break
        n = n * 26 + ord(ch.upper()) - 64
    return n - 1


def leer_excel_streaming(path):
    """
    Lee solo las columnas necesarias directamente del XML del .xlsx, fila por fila.
    pandas.read_excel cargaría las 121 columnas x 578 mil filas y agotaría la RAM.
    Devuelve un DataFrame de texto (dtype=str), con las filas en el orden original.
    """
    print(f"Leyendo {os.path.basename(path)} (puede tardar varios minutos)...")
    t0 = time.time()
    z = zipfile.ZipFile(path)

    shared = []
    with z.open("xl/sharedStrings.xml") as f:
        for _, el in ET.iterparse(f):
            if el.tag == _NS + "si":
                shared.append("".join(t.text or "" for t in el.iter(_NS + "t")))
                el.clear()

    hojas = sorted(n for n in z.namelist()
                   if n.startswith("xl/worksheets/sheet") and n.endswith(".xml"))
    filas, idx_nombre = [], None
    with z.open(hojas[0]) as f:
        for _, row in ET.iterparse(f):
            if row.tag != _NS + "row":
                continue
            celdas = {}
            for c in row:
                v = c.find(_NS + "v")
                t = c.get("t")
                if t == "inlineStr":
                    val = "".join(x.text or "" for x in c.iter(_NS + "t"))
                elif v is None or v.text is None:
                    continue
                elif t == "s":
                    val = shared[int(v.text)]
                else:
                    val = v.text
                celdas[_col_idx(c.get("r"))] = val
            if idx_nombre is None:      # primera fila = encabezados
                nombres = {i: s.strip() for i, s in celdas.items()}
                inv = {s: i for i, s in nombres.items()}
                faltan = [c for c in COLUMNAS if c not in inv]
                if faltan:
                    raise RuntimeError(f"Faltan columnas en el Excel: {faltan}")
                idx_nombre = [inv[c] for c in COLUMNAS]
            else:
                filas.append([celdas.get(i) for i in idx_nombre])
                if len(filas) % 100000 == 0:
                    print(f"   {len(filas):,} filas leídas...")
            row.clear()

    df = pd.DataFrame(filas, columns=COLUMNAS, dtype=object)
    print(f"✓ Excel leído: {len(df):,} filas en {time.time() - t0:.0f} s")
    return df


def cargar_dataset():
    if USAR_CACHE and os.path.exists(CACHE_PATH) and \
            os.path.getmtime(CACHE_PATH) >= os.path.getmtime(EXCEL_PATH):
        print("✓ Usando copia en caché del Excel")
        return pd.read_pickle(CACHE_PATH)
    df = leer_excel_streaming(EXCEL_PATH)
    if USAR_CACHE:
        df.to_pickle(CACHE_PATH)
    return df


# --------------------------------------------------------- transformación
DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]


def limpiar(df):
    """Tipos correctos. NO elimina ninguna fila (la fact debe tener las mismas filas que el Excel)."""
    df = df.copy()
    for c in COLS_ENTERAS:
        df[c] = pd.to_numeric(df[c], errors="raise").astype("int64")
    for c in MEDIDAS:
        df[c] = pd.to_numeric(df[c], errors="raise").astype("float64").round(4)
    for c in DEGENERADAS + ["placa", "vendedorinterno", "destinofinal"]:
        df[c] = df[c].fillna("").astype(str).str.strip()
    df["estadoalmacen"] = df["estadoalmacen"].astype(str)
    if df[COLS_ENTERAS + MEDIDAS].isna().any().any():
        raise ValueError("Hay nulos en columnas numéricas del Excel.")
    return df


def _dim(df, claves, nombre_id):
    """Dimensión = combinaciones únicas de `claves`, con clave sustituta 1..n (orden estable)."""
    d = df[claves].drop_duplicates().sort_values(claves).reset_index(drop=True)
    d.insert(0, nombre_id, np.arange(1, len(d) + 1, dtype="int64"))
    return d


def construir_modelo(df):
    """Devuelve (dict de dimensiones, fact). Valida que la fact conserve todas las filas."""
    n = len(df)

    # --- DIM_TIEMPO (a partir de iddia = yyyymmdd) ---
    fechas = pd.to_datetime(df["iddia"].astype(str), format="%Y%m%d")
    df = df.assign(_fecha=fechas.dt.normalize())
    t = _dim(df, ["_fecha"], "id_tiempo")
    f = t["_fecha"]
    tiempo = pd.DataFrame({
        "id_tiempo": t["id_tiempo"],
        "iddia": f.dt.date,
        "anio": f.dt.year.astype("int64"),
        "trimestre": f.dt.quarter.astype("int64"),
        "mes": f.dt.month.astype("int64"),
        "semana": f.dt.isocalendar().week.astype("int64"),
        "dia_semana": f.dt.dayofweek.map(lambda i: DIAS[i]),
    })

    # --- Resto de dimensiones. Se define su grano por las columnas que el diagrama
    #     muestra en cada una (hay claves naturales con atributos que cambian entre filas,
    #     p.ej. un mismo idmaterialcom con 2 unidades de medida).
    # Atributos descriptivos que NO existen en el Excel (nombres, familias, zonas...) quedan en NULL.
    cliente = _dim(df, ["idcliente"], "id_cliente")
    cliente["tipo_cliente"] = None
    cliente["segmento"] = None

    destinatario = _dim(df, ["iddestinatario"], "id_destinatario")
    destinatario["cod_destinatario"] = None
    destinatario["nombre_destinatario"] = None

    producto = _dim(df, ["idmaterialcom", "idunidadmedida"], "id_producto")
    producto["desmaterialcom"] = None
    producto["familia"] = None
    producto["linea"] = None

    oficina = _dim(df, ["idoficina"], "id_oficina")
    oficina["nombre_oficina"] = None
    oficina["zona"] = None

    # zona_destino se alimenta con la columna destinofinal del Excel (Trujillo, Piura, ...)
    ruta = _dim(df, ["idruta", "destinofinal"], "id_ruta").rename(columns={"destinofinal": "zona_destino"})
    ruta["descripcion_ruta"] = None

    transporte = _dim(df, ["idempresatransporte", "idtipovehiculo", "placa"], "id_transporte")
    transporte["nombre_empresa"] = None

    vendedor = _dim(df, ["codvendedorinterno", "vendedorinterno", "idgrupovendedor"], "id_vendedor")

    canal = _dim(df, ["idsubcanal"], "id_canal")
    canal["desc_canal"] = None

    tipo_desp = _dim(df, ["idtipodespacho", "idclasedespacho", "idestadoentrega", "estadoalmacen"],
                     "id_tipo_despacho")

    unidad = _dim(df, ["idunidadmedida"], "id_unidad_medida")
    unidad["desc_unidad"] = None

    # --- FACT: un LEFT JOIN m:1 por dimensión; no puede multiplicar ni perder filas ---
    fact = df.copy()
    fact.insert(0, "id_despacho", np.arange(1, n + 1, dtype="int64"))
    fact = fact.merge(t[["_fecha", "id_tiempo"]], on="_fecha", how="left", validate="m:1")
    uniones = [
        (cliente, ["idcliente"], "id_cliente"),
        (destinatario, ["iddestinatario"], "id_destinatario"),
        (producto, ["idmaterialcom", "idunidadmedida"], "id_producto"),
        (oficina, ["idoficina"], "id_oficina"),
        (ruta.rename(columns={"zona_destino": "destinofinal"}), ["idruta", "destinofinal"], "id_ruta"),
        (transporte, ["idempresatransporte", "idtipovehiculo", "placa"], "id_transporte"),
        (vendedor, ["codvendedorinterno", "vendedorinterno", "idgrupovendedor"], "id_vendedor"),
        (canal, ["idsubcanal"], "id_canal"),
        (tipo_desp, ["idtipodespacho", "idclasedespacho", "idestadoentrega", "estadoalmacen"], "id_tipo_despacho"),
        (unidad, ["idunidadmedida"], "id_unidad_medida"),
    ]
    for dim, claves, id_col in uniones:
        fact = fact.merge(dim[claves + [id_col]], on=claves, how="left", validate="m:1")

    cols_fact = (["id_despacho", "id_tiempo", "id_cliente", "id_destinatario", "id_producto",
                  "id_oficina", "id_ruta", "id_transporte", "id_vendedor", "id_canal",
                  "id_tipo_despacho", "id_unidad_medida"] + DEGENERADAS + MEDIDAS)
    fact = fact[cols_fact]

    if len(fact) != n:
        raise RuntimeError(f"La fact tiene {len(fact)} filas y el Excel {n}.")
    if fact.isna().any()[[c for c in cols_fact if c.startswith("id_")]].any():
        raise RuntimeError("Hay filas de la fact sin clave de alguna dimensión.")

    dims = {
        "DIM_TIEMPO": tiempo, "DIM_CLIENTE": cliente, "DIM_DESTINATARIO": destinatario,
        "DIM_PRODUCTO": producto, "DIM_OFICINA": oficina, "DIM_RUTA": ruta,
        "DIM_TRANSPORTE": transporte, "DIM_VENDEDOR": vendedor, "DIM_CANAL": canal,
        "DIM_TIPO_DESPACHO": tipo_desp, "DIM_UNIDAD_MEDIDA": unidad,
    }
    # orden de columnas igual al del CREATE TABLE
    orden = {
        "DIM_TIEMPO": ["id_tiempo", "iddia", "anio", "trimestre", "mes", "semana", "dia_semana"],
        "DIM_CLIENTE": ["id_cliente", "idcliente", "tipo_cliente", "segmento"],
        "DIM_DESTINATARIO": ["id_destinatario", "iddestinatario", "cod_destinatario", "nombre_destinatario"],
        "DIM_PRODUCTO": ["id_producto", "idmaterialcom", "desmaterialcom", "idunidadmedida", "familia", "linea"],
        "DIM_OFICINA": ["id_oficina", "idoficina", "nombre_oficina", "zona"],
        "DIM_RUTA": ["id_ruta", "idruta", "descripcion_ruta", "zona_destino"],
        "DIM_TRANSPORTE": ["id_transporte", "idempresatransporte", "nombre_empresa", "idtipovehiculo", "placa"],
        "DIM_VENDEDOR": ["id_vendedor", "codvendedorinterno", "vendedorinterno", "idgrupovendedor"],
        "DIM_CANAL": ["id_canal", "idsubcanal", "desc_canal"],
        "DIM_TIPO_DESPACHO": ["id_tipo_despacho", "idtipodespacho", "idclasedespacho", "idestadoentrega", "estadoalmacen"],
        "DIM_UNIDAD_MEDIDA": ["id_unidad_medida", "idunidadmedida", "desc_unidad"],
    }
    dims = {k: v[orden[k]] for k, v in dims.items()}
    return dims, fact


# ------------------------------------------------------------------ carga
def insertar_df(cursor, tabla, df):
    """INSERT masivo por lotes (fast_executemany). Convierte a tipos nativos de Python."""
    cols = list(df.columns)
    sql = f"INSERT INTO {tabla} ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))})"
    cursor.fast_executemany = True
    total = len(df)
    for ini in range(0, total, BATCH):
        lote = df.iloc[ini:ini + BATCH].astype(object)
        lote = lote.where(pd.notna(lote), None)
        cursor.executemany(sql, list(lote.itertuples(index=False, name=None)))
        if total > BATCH:
            print(f"   {tabla}: {min(ini + BATCH, total):,}/{total:,}")
    return total


def poblar(conn, dims, fact):
    cur = conn.cursor()
    for nombre, d in dims.items():              # primero dimensiones
        insertar_df(cur, nombre, d)
        conn.commit()
        print(f"✓ {nombre}: {len(d):,} filas")
    insertar_df(cur, "FACT_DESPACHOS", fact)    # luego la fact
    conn.commit()
    print(f"✓ FACT_DESPACHOS: {len(fact):,} filas")


def validar(conn, df_excel, fact):
    """Compara lo cargado en SQL contra el Excel: nº de filas y totales de medidas."""
    cur = conn.cursor()
    n_sql = cur.execute("SELECT COUNT(*) FROM FACT_DESPACHOS").fetchone()[0]
    print(f"\nFilas en el Excel     : {len(df_excel):,}")
    print(f"Filas en FACT_DESPACHOS: {n_sql:,}")
    if n_sql != len(df_excel):
        raise RuntimeError("¡La fact NO tiene la misma cantidad de filas que el Excel!")
    sumas = ", ".join(f"SUM({m})" for m in MEDIDAS)
    fila = cur.execute(f"SELECT {sumas} FROM FACT_DESPACHOS").fetchone()
    print(f"{'medida':22s}{'Excel':>22s}{'SQL':>22s}")
    for m, s_sql in zip(MEDIDAS, fila):
        s_xl = float(df_excel[m].sum())
        print(f"{m:22s}{s_xl:22,.2f}{float(s_sql):22,.2f}")
        if abs(s_xl - float(s_sql)) > 0.05:      # solo ruido de coma flotante
            raise RuntimeError(f"La suma de {m} no coincide entre Excel y SQL.")
    print("✓ Validación correcta: misma cantidad de filas y mismos totales.")


# --------------------------------------------------------------------- main
def main():
    df = limpiar(cargar_dataset())
    print(f"Dataset: {len(df):,} filas")
    dims, fact = construir_modelo(df)
    for k, v in dims.items():
        print(f"  {k:20s} {len(v):>6,} filas")
    print(f"  {'FACT_DESPACHOS':20s} {len(fact):>6,} filas (debe ser igual al Excel)")

    crear_base_si_no_existe()
    conn = get_connection(DB_DESTINO)
    if conn is None:
        sys.exit(1)
    try:
        crear_esquema(conn)
        poblar(conn, dims, fact)
        validar(conn, df, fact)
    finally:
        conn.close()
    print("ETL completado. Data Mart CEMEX poblado correctamente.")


if __name__ == "__main__":
    main()

Leyendo Dataset_CEMEX.xlsx (puede tardar varios minutos)...
   100,000 filas leídas...
   200,000 filas leídas...
   300,000 filas leídas...
   400,000 filas leídas...
   500,000 filas leídas...
✓ Excel leído: 578,234 filas en 584 s
Dataset: 578,234 filas
  DIM_TIEMPO              722 filas
  DIM_CLIENTE           5,703 filas
  DIM_DESTINATARIO     13,595 filas
  DIM_PRODUCTO          1,044 filas
  DIM_OFICINA              20 filas
  DIM_RUTA                840 filas
  DIM_TRANSPORTE        1,989 filas
  DIM_VENDEDOR            949 filas
  DIM_CANAL                 7 filas
  DIM_TIPO_DESPACHO        34 filas
  DIM_UNIDAD_MEDIDA        10 filas
  FACT_DESPACHOS       578,234 filas (debe ser igual al Excel)
✓ Conexión a master exitosa
✓ Conexión a CemexDW exitosa
✓ Esquema del Data Mart creado (11 dimensiones + FACT_DESPACHOS).
✓ DIM_TIEMPO: 722 filas
✓ DIM_CLIENTE: 5,703 filas
✓ DIM_DESTINATARIO: 13,595 filas
✓ DIM_PRODUCTO: 1,044 filas
✓ DIM_OFICINA: 20 filas
✓ DIM_RUTA: 840 filas
✓ DI